In [ ]:
import importlib.util, subprocess, sys
_missing = [p for p in ('huggingface_hub', 'tqdm', 'pandas', 'matplotlib') if importlib.util.find_spec(p) is None]
if _missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_missing], check=True)
    print('installed', _missing, '- now Restart the kernel once, then Run All')
else:
    print('all packages present')

In [ ]:
import os, re, json, math, random, hashlib, zipfile, gc, time, pickle, warnings, sys, copy, shutil
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

SMOKE_TEST = False
SEED = 42
RESUME_COMPLETED_RUNS = True
WRITE_SUBMISSIONS = True
RUN_SECTION_ANALYSIS = True
SECTION_ANALYSIS_DOCS = 2000

DATASET_ID = "Exploration-Lab/CS-781-Capstone"
DATA_SOURCE = "hf"
LOCAL_DATA_DIR = None
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working", "."]
PHASE1_CKPT_OVERRIDE = None
TOKENIZER_PATH_OVERRIDE = None

CHUNK_TOKENS = 512
N_CHUNKS = 4
SELECTION = "spread"
HEAD_CHUNKS = 1
EVAL_N_CHUNKS = None
TRAIN_MAX_WORDS = 8000
EVAL_MAX_WORDS = 60000
XL_MEMORY = 0
XL_MEMORY_OLD = 0
CHUNK_POOLING = "mean+last"
HEAD = "pooled"

LABEL_CORRELATION = True
HIDDEN = 512
N_POS_BUCKETS = 8

LOSS = "asl"
ASL_GAMMA_NEG, ASL_GAMMA_POS, ASL_CLIP = 4.0, 0.0, 0.05
POS_WEIGHT_CAP = 20.0
RARE_BOOST = 3.0
MIN_DEV_SUPPORT_PER_LABEL = 3
AT_LEAST_ONE = True

EPOCHS = 5
TRAIN_DOCS_PER_EPOCH = 12000
EPOCH_DEV_DOCS = 3000
MAX_TRAIN_MINUTES = 60
BATCH_DOCS = 4
GRAD_ACCUM = 4
EVAL_BATCH_DOCS = 8
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.06
GRAD_CLIP = 1.0
HEAD_DROPOUT = 0.1
BACKBONE_DROPOUT = 0.1
USE_GRAD_CHECKPOINTING = False

EXPERIMENT_LIBRARY = {
    "lorma_plus": dict(method="lorma_plus", rank=16, alpha=16, lr=5e-4, head_lr=2e-3, patience=2,
                       description="LoRMA+ (compulsory): multiplicative low-rank update W = (I + a/r * B A) W0."),
    "qlora": dict(method="qlora", rank=16, alpha=32, lr=1e-3, head_lr=2e-3, patience=2,
                  description="QLoRA: frozen Phase-1 weights stored in 4-bit NF4 (block 64, absmax) + LoRA on top."),
    "adalora": dict(method="adalora", init_rank=12, target_rank=4, alpha=16, orth_reg=0.1, beta1=0.85, beta2=0.85,
                    t_init=0.10, t_final=0.20, delta_t=10, lr=1e-3, head_lr=2e-3, patience=2,
                    description="AdaLoRA: SVD-form update P diag(E) Q with an importance-driven rank budget (12 -> 4 avg)."),
    "adapter": dict(method="adapter", bottleneck=64, lr=1e-3, head_lr=2e-3, patience=2,
                    description="Houlsby adapters: bottleneck MLP after every attention output and every FFN (+ norms)."),
    "partial": dict(method="partial", top_layers=4, llrd=0.85, lr=5e-5, head_lr=2e-3, patience=2,
                    description="Partial / layer-wise fine-tuning: top 4 layers trained, layer-wise LR decay 0.85."),
    "prefix": dict(method="prefix", prefix_len=32, prefix_hidden=256, lr=1e-3, head_lr=2e-3, patience=2,
                   description="Prefix tuning: trainable per-layer prefix states (MLP re-parameterised) in every layer's memory."),
    "frozen": dict(method="frozen", lr=0.0, head_lr=2e-3, patience=2,
                   description="Frozen Phase-1 backbone, only the task head trained (how useful are the Phase-1 representations?)."),
    "full": dict(method="full", lr=3e-5, head_lr=2e-3, patience=2,
                 description="Full fine-tuning (reference upper bound)."),
    "distill": dict(method="distill", kd_temperature=2.0, kd_alpha=0.5, kd_hidden_weight=0.5, lr=5e-5, head_lr=2e-3,
                    patience=2, description="Knowledge distillation: fine-tuned teacher -> smaller student."),
}
EXPERIMENT_DEFAULTS = dict(selection=None, n_chunks=None, head=None, loss=None, label_correlation=None, epochs=None,
                           dropout=None, use_amp=True)

def make_experiment(base, **over):
    cfg = copy.deepcopy(EXPERIMENT_LIBRARY[base])
    cfg.update(over)
    return cfg

ARCH_NAME = "transformer_xl"
OUTPUT_NAME = "phase5_transformer_xl_lsi"
MIN_PARAMS, MAX_PARAMS = 50e6, 300e6
XL_MEMORY = 256
PHASE1_ARCH_NAME = "transformer_xl_relative_recurrence_tuned"
LOWRANK_TARGETS = ("attn.q_net", "attn.k_net", "attn.v_net", "attn.o_net", "ff.net.0", "ff.net.2")
ADAPTER_TARGETS = ("attn.o_net", "ff.net.2")
EXPERIMENTS = {
    "E3_partial_ft": make_experiment("partial"),
    "E4_distill": make_experiment("distill", student_layers=6,
                                  description="KD: best fine-tuned 12-layer teacher (~99.8M) -> 6-layer Transformer-XL student (~53.8M)."),
}
RUN_LIST = ["E3_partial_ft", "E4_distill"]

OUTPUT_DIR = Path("/kaggle/working") / OUTPUT_NAME if Path("/kaggle/working").exists() else Path(OUTPUT_NAME)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "runs").mkdir(exist_ok=True)

if SMOKE_TEST:
    CHUNK_TOKENS, N_CHUNKS, HIDDEN, TRAIN_DOCS_PER_EPOCH = 32, 3, 32, None
    BATCH_DOCS, GRAD_ACCUM, EVAL_BATCH_DOCS, EPOCHS = 2, 1, 4, 2
    XL_MEMORY = min(XL_MEMORY, 16)
    XL_MEMORY_OLD = min(XL_MEMORY_OLD, 16)
    MIN_PARAMS = 0
    for _cfg in EXPERIMENTS.values():
        if "delta_t" in _cfg:
            _cfg["delta_t"] = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          "| memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
elif not SMOKE_TEST:
    raise RuntimeError("Enable a GPU accelerator (Kaggle: Settings > Accelerator > GPU T4/P100).")
print("Experiments to run:", RUN_LIST)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

def save_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)

def make_grad_scaler(enabled):
    if not enabled:
        return None
    try:
        return torch.amp.GradScaler("cuda", enabled=True)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=True)

In [ ]:
class ByteBPE:
    EOW = 260

    def __init__(self):
        self.specials = ["<pad>", "<unk>", "<s>", "</s>"]
        self.itos, self.stoi, self.payload, self.merges = [], {}, {}, []
        self.vocab_size = 0
        self._ranks = {}
        self._cache = {}

    @classmethod
    def from_json(cls, payload):
        t = cls()
        t.itos = t.specials + [f"<0x{i:02X}>" for i in range(256)] + ["</w>"]
        t.stoi = {s: i for i, s in enumerate(t.itos)}
        t.payload = {i: bytes([i - 4]) for i in range(4, 260)}
        t.payload[260] = b""
        t.merges = [tuple(map(int, p)) for p in payload["merges"]]
        for pair in t.merges:
            new_id = len(t.itos)
            symbol = f"<{pair[0]}+{pair[1]}>"
            t.itos.append(symbol)
            t.stoi[symbol] = new_id
            t.payload[new_id] = t.payload.get(pair[0], b"") + t.payload.get(pair[1], b"")
        t.vocab_size = len(t.itos)
        if t.itos != payload.get("itos"):
            raise ValueError("Saved ByteBPE vocabulary and merge list are inconsistent.")
        t._ranks = {pair: r for r, pair in enumerate(t.merges)}
        return t

    def encode_word_reference(self, word):
        seq = [b + 4 for b in str(word).encode("utf-8")] + [self.EOW]
        for rank, pair in enumerate(self.merges):
            out, j = [], 0
            while j < len(seq):
                if j + 1 < len(seq) and (seq[j], seq[j + 1]) == pair:
                    out.append(261 + rank); j += 2
                else:
                    out.append(seq[j]); j += 1
            seq = out
        return seq

    def encode_piece(self, piece, add_eow=True):
        key = (piece, add_eow)
        hit = self._cache.get(key)
        if hit is not None:
            return hit
        seq = [b + 4 for b in piece.encode("utf-8")]
        if add_eow:
            seq.append(self.EOW)
        ranks = self._ranks
        while len(seq) > 1:
            best_rank, best_pair = None, None
            for i in range(len(seq) - 1):
                r = ranks.get((seq[i], seq[i + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_pair = r, (seq[i], seq[i + 1])
            if best_pair is None:
                break
            out, j, new_id = [], 0, 261 + best_rank
            while j < len(seq):
                if j + 1 < len(seq) and (seq[j], seq[j + 1]) == best_pair:
                    out.append(new_id); j += 2
                else:
                    out.append(seq[j]); j += 1
            seq = out
        if len(self._cache) < 400_000:
            self._cache[key] = seq
        return seq

def _sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def _search(patterns, roots):
    found, seen = [], set()
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for pat in patterns:
            for p in sorted(root.rglob(pat)):
                rp = str(p.resolve())
                if p.is_file() and rp not in seen:
                    seen.add(rp); found.append(p)
    return found

if PHASE1_CKPT_OVERRIDE:
    ckpt_candidates = [Path(PHASE1_CKPT_OVERRIDE)]
else:
    ckpt_candidates = _search(["best_checkpoint.pt", "best*checkpoint*.pt"], SEARCH_ROOTS) + \
                      _search(["latest*checkpoint*.pt", "*.pt"], SEARCH_ROOTS)
_payload, PHASE1_CKPT_PATH = None, None
for cand in ckpt_candidates:
    try:
        pl = torch.load(cand, map_location="cpu", weights_only=False)
    except Exception as exc:
        print(f"  skip {cand}: {exc}"); continue
    arch = str(pl.get("architecture", "")) if isinstance(pl, dict) else ""
    if isinstance(pl, dict) and "model_state_dict" in pl and arch == PHASE1_ARCH_NAME:
        _payload, PHASE1_CKPT_PATH = pl, cand
        break
    print(f"  skip {cand}: architecture={arch!r}")
    del pl
if _payload is None:
    raise FileNotFoundError("No standard Transformer-XL Phase-1 checkpoint (architecture transformer_xl_relative_recurrence_tuned) "
                            "found. Attach the transformer_xl_tuned_100m output with best_checkpoint.pt, or set PHASE1_CKPT_OVERRIDE.")
print("Phase-1 checkpoint:", PHASE1_CKPT_PATH)
P1_STATE = _payload["model_state_dict"]
P1_CFG = dict(_payload["model_config"])
P1_VOCAB = int(_payload.get("vocab_size", P1_CFG["vocab_size"]))
P1_TOKENIZER_SHA = _payload.get("tokenizer_sha256")
P1_ARCH = _payload.get("architecture")
P1_CFG.setdefault("rel_base", 10_000.0)
print("Phase-1 architecture:", P1_ARCH)
print("Phase-1 config:", {k: P1_CFG.get(k) for k in ("d_model", "heads", "layers", "ff_hidden", "training_segment_len",
                                                     "training_mem_len", "vocab_size")})
del _payload
gc.collect()

if TOKENIZER_PATH_OVERRIDE:
    tok_candidates = [Path(TOKENIZER_PATH_OVERRIDE)]
else:
    tok_candidates = _search(["legal_bpe_tokenizer*.json", "*bpe*.json", "*tokenizer*.json"], SEARCH_ROOTS)

TOKENIZER, TOKENIZER_PATH, TOKENIZER_SHA256 = None, None, None
tok_table = []
for cand in tok_candidates:
    try:
        raw = json.loads(cand.read_text(encoding="utf-8"))
        if not (isinstance(raw, dict) and "merges" in raw and "itos" in raw):
            continue
        sha = _sha256_file(cand)
        tok_table.append((cand, int(raw.get("vocab_size", len(raw["itos"]))), sha, raw))
    except Exception:
        continue

print("Tokenizer files found:")
for cand, vs, sha, _ in tok_table:
    print(f"  {cand}  vocab={vs}  sha256={sha[:12]}")

chosen = [t for t in tok_table if P1_TOKENIZER_SHA and t[2] == P1_TOKENIZER_SHA]
how = "sha256 match with the Phase-1 checkpoint"
if not chosen:
    chosen = [t for t in tok_table if t[1] == P1_VOCAB]
    how = f"vocab size match ({P1_VOCAB}); sha256 differs/unknown - verify this is your Phase-1 tokenizer file"
if not chosen:
    raise FileNotFoundError(
        f"No tokenizer with vocab size {P1_VOCAB} found. Attach your legal_bpe_tokenizer JSON for that size "
        "(or set TOKENIZER_PATH_OVERRIDE). This notebook never trains a tokenizer.")
TOKENIZER_PATH, _, TOKENIZER_SHA256, _raw = chosen[0]
TOKENIZER = ByteBPE.from_json(_raw)
del tok_table, _raw
print(f"Using tokenizer: {TOKENIZER_PATH} ({how})")

PAD_ID, UNK_ID, BOS_ID, EOS_ID = (TOKENIZER.stoi[s] for s in ("<pad>", "<unk>", "<s>", "</s>"))
EOW_ID = ByteBPE.EOW
VOCAB_SIZE = TOKENIZER.vocab_size
assert VOCAB_SIZE == P1_VOCAB, f"Tokenizer vocab {VOCAB_SIZE} != checkpoint vocab {P1_VOCAB}"
print("Vocab size:", VOCAB_SIZE, "| ids:", dict(pad=PAD_ID, unk=UNK_ID, bos=BOS_ID, eos=EOS_ID, eow=EOW_ID))

_rng = random.Random(0)
_words = ["The", "Supreme", "Court", "of", "India,", "Section", "302", "IPC.", "(1995)", "Crl.A.No.123/2005",
          "appellant-State", "Hon'ble", "₹5,000", "naïve"]
for _w in _words:
    assert TOKENIZER.encode_piece(_w, True) == TOKENIZER.encode_word_reference(_w), _w
print("Fast BPE encoder == Phase-1 reference encoder on self-check words.")

In [ ]:
SPLIT_FILES = {"train": "train.json", "dev": "dev.json", "test": "test.json"}
LABEL_KEYS = ("labels", "label", "statutes", "statute_ids", "y", "targets", "gold")
PHASE_RE = r"phase[_ -]?5|lsi"

def _pick(hits):
    pref = [h for h in hits if re.search(PHASE_RE, str(h), re.I)]
    return (pref or hits)[0]

def locate_phase5_files():
    paths = {}
    if DATA_SOURCE == "local":
        roots = [LOCAL_DATA_DIR] if LOCAL_DATA_DIR else SEARCH_ROOTS
        for split, name in SPLIT_FILES.items():
            hits = _search([name], roots)
            if not hits:
                raise FileNotFoundError(f"{name} not found under {roots}")
            paths[split] = _pick(hits)
        return paths
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
    if not token:
        from getpass import getpass
        token = getpass("Hugging Face read token: ")
    from huggingface_hub import HfApi, hf_hub_download
    repo_files = HfApi().list_repo_files(DATASET_ID, repo_type="dataset", token=token)
    for split, name in SPLIT_FILES.items():
        hits = [f for f in repo_files if f.endswith("/" + name) or f == name]
        ph = [f for f in hits if re.search(PHASE_RE, f, re.I)]
        if not ph:
            raise FileNotFoundError(f"Phase-5 {name} not found in {DATASET_ID}. Candidates: {hits[:10]} | "
                                    f"some files: {repo_files[:30]}")
        paths[split] = Path(hf_hub_download(DATASET_ID, ph[0], repo_type="dataset", token=token))
    return paths

P5_PATHS = locate_phase5_files()
for k, v in P5_PATHS.items():
    print(f"{k:5s} -> {v}")

def _records(obj):
    if isinstance(obj, dict):
        for key in ("data", "documents", "docs", "cases", "examples"):
            if key in obj and isinstance(obj[key], list):
                return obj[key]
        out = []
        for k, v in obj.items():
            if isinstance(v, dict):
                v = dict(v); v.setdefault("id", k); out.append(v)
            elif isinstance(v, str):
                out.append({"id": k, "text": v})
        return out
    return obj

def _ids(v):
    if v is None:
        return []
    if isinstance(v, (int, np.integer, str)):
        v = [v]
    out = []
    for x in v:
        if isinstance(x, bool):
            raise ValueError(f"unexpected statute id {x!r}")
        if isinstance(x, (int, np.integer)) or (isinstance(x, float) and float(x).is_integer()):
            out.append(int(x))
        elif isinstance(x, str) and x.strip().lstrip("-").isdigit():
            out.append(int(x.strip()))
        else:
            raise ValueError(f"statute id {x!r} is not an integer - the submission needs integer ids")
    return sorted(set(out))

def load_split(path, with_labels):
    raw = path.read_text(encoding="utf-8")
    try:
        obj = json.loads(raw)
    except json.JSONDecodeError:
        obj = [json.loads(line) for line in raw.splitlines() if line.strip()]
    docs = []
    for i, d in enumerate(_records(obj)):
        did = d.get("id", d.get("case_id", d.get("doc_id", f"case_{i}")))
        text = d.get("text", d.get("document", d.get("facts", "")))
        if isinstance(text, list):
            text = " ".join("" if t is None else str(t) for t in text)
        doc = {"id": did, "text": "" if text is None else str(text)}
        if with_labels:
            key = next((k for k in LABEL_KEYS if k in d), None)
            if key is None:
                raise KeyError(f"{did}: no statute field (looked for {LABEL_KEYS}); keys = {list(d)}")
            doc["statutes"] = _ids(d[key])
        docs.append(doc)
    return docs

DATA = {s: load_split(P5_PATHS[s], with_labels=(s != "test")) for s in SPLIT_FILES}
test_ids = [d["id"] for d in DATA["test"]]
assert len(set(map(str, test_ids))) == len(test_ids), "duplicate test ids"

LABEL_IDS = sorted({s for split in ("train", "dev") for d in DATA[split] for s in d["statutes"]})
LABEL2IDX = {s: i for i, s in enumerate(LABEL_IDS)}
NUM_LABELS = len(LABEL_IDS)
for split in ("train", "dev"):
    for d in DATA[split]:
        y = np.zeros(NUM_LABELS, np.float32)
        y[[LABEL2IDX[s] for s in d["statutes"]]] = 1.0
        d["y"] = y
Y_TRAIN = np.stack([d["y"] for d in DATA["train"]])
Y_DEV = np.stack([d["y"] for d in DATA["dev"]])
TRAIN_LABEL_COUNTS = Y_TRAIN.sum(0)
only_dev = [LABEL_IDS[i] for i in np.where(TRAIN_LABEL_COUNTS == 0)[0]]

for s, docs in DATA.items():
    words = np.array([len(d["text"].split()) for d in docs])
    extra = ""
    if s != "test":
        k = np.array([len(d["statutes"]) for d in docs])
        extra = f" | statutes/doc: mean {k.mean():.2f}, max {k.max()}, docs with >1: {np.mean(k > 1):.1%}"
    print(f"[{s:5s}] {len(docs):6,d} documents | words: median {int(np.median(words)):,} p90 "
          f"{int(np.percentile(words, 90)):,} max {words.max():,}{extra}")
print(f"{NUM_LABELS} statute ids (train + dev), range {LABEL_IDS[0]}..{LABEL_IDS[-1]}"
      + (f" | {len(only_dev)} appear only in dev: {only_dev[:10]}" if only_dev else ""))
_c = np.sort(TRAIN_LABEL_COUNTS)[::-1]
print(f"train positives per statute: max {int(_c[0])}, median {int(np.median(_c))}, min {int(_c[-1])} | "
      f"statutes with < 20 training documents: {int((TRAIN_LABEL_COUNTS < 20).sum())}")

_q = np.quantile(TRAIN_LABEL_COUNTS, [1 / 3, 2 / 3])
FREQ_BUCKET = np.where(TRAIN_LABEL_COUNTS <= _q[0], "rare", np.where(TRAIN_LABEL_COUNTS <= _q[1], "medium", "frequent"))

In [ ]:
TOKEN_DTYPE = np.uint16 if VOCAB_SIZE <= 65535 else np.int32

def tokenise_doc(text, max_words):
    words = text.split()
    n_words = len(words)
    if n_words > max_words:
        h = max_words // 2
        words = words[:h] + words[-(max_words - h):]
    ids = []
    enc = TOKENIZER.encode_piece
    for w in words:
        ids.extend(enc(w, True))
    return np.asarray(ids or [UNK_ID], dtype=TOKEN_DTYPE), n_words

_tok_cache = OUTPUT_DIR / (f"p5_tokens_{TOKENIZER_SHA256[:8]}_{TRAIN_MAX_WORDS}_{EVAL_MAX_WORDS}_"
                           f"{sum(len(v) for v in DATA.values())}.pkl")
if _tok_cache.exists():
    with _tok_cache.open("rb") as f:
        _cached = pickle.load(f)
    for split in DATA:
        for d in DATA[split]:
            d["stream"], d["n_words"] = _cached[(split, d["id"])]
    print("Loaded cached tokenisation.")
else:
    _cached = {}
    for split in DATA:
        mw = TRAIN_MAX_WORDS if split == "train" else EVAL_MAX_WORDS
        for d in tqdm(DATA[split], desc=f"tokenising {split}"):
            d["stream"], d["n_words"] = tokenise_doc(d["text"], mw)
            _cached[(split, d["id"])] = (d["stream"], d["n_words"])
    with _tok_cache.open("wb") as f:
        pickle.dump(_cached, f, protocol=4)
for split in DATA:
    for d in DATA[split]:
        d["n_tokens"] = len(d["stream"])
_dl = np.array([d["n_tokens"] for d in DATA["dev"]])
print(f"dev documents: tokens median {int(np.median(_dl)):,} | p90 {int(np.percentile(_dl, 90)):,} | "
      f"longer than {N_CHUNKS} x {CHUNK_TOKENS} = {N_CHUNKS * CHUNK_TOKENS:,} tokens: {np.mean(_dl > N_CHUNKS * CHUNK_TOKENS):.1%}")

def select_starts(n, mode, K, head_k=HEAD_CHUNKS, C=None):
    C = C or CHUNK_TOKENS
    if n <= C:
        return [0]
    if mode == "all" or math.ceil(n / C) <= K:
        st = list(range(0, n - C, C)) + [n - C]
        if mode != "all" and len(st) > K:
            st = [int(round(x)) for x in np.linspace(0, n - C, K)]
        return sorted(set(st))
    if mode == "head":
        st = [i * C for i in range(K)]
    elif mode == "tail":
        st = [n - (K - i) * C for i in range(K)]
    elif mode == "head+tail":
        h = min(head_k, K - 1)
        st = [i * C for i in range(h)] + [n - (K - h - j) * C for j in range(K - h)]
    elif mode == "middle":
        s0 = int(min(max(0, n / 2 - K * C / 2), n - K * C))
        st = [s0 + i * C for i in range(K)]
    elif mode == "spread":
        st = [int(round(x)) for x in np.linspace(0, n - C, K)]
    else:
        raise ValueError(mode)
    return sorted(set(max(0, min(s, n - C)) for s in st))

def make_doc_item(doc, mode, K, with_label=True):
    n = doc["n_tokens"]
    chunks = []
    for a in select_starts(n, mode, K):
        b = min(n, a + CHUNK_TOKENS)
        mem = doc["stream"][max(0, a - XL_MEMORY - XL_MEMORY_OLD):a] if XL_MEMORY else np.zeros(0, TOKEN_DTYPE)
        bucket = min(N_POS_BUCKETS - 1, int((a + b) / 2 / max(n, 1) * N_POS_BUCKETS))
        chunks.append(dict(ids=doc["stream"][a:b].astype(np.int64), start=a, mem=mem.astype(np.int64), bucket=bucket))
    item = dict(chunks=chunks, id=doc["id"])
    if with_label and "y" in doc:
        item["y"] = doc["y"]
    return item

class DocDataset(Dataset):
    def __init__(self, docs, mode, K):
        self.docs, self.mode, self.K = docs, mode, K

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, i):
        return make_doc_item(self.docs[i], self.mode, self.K)

_mx = TRAIN_LABEL_COUNTS.max()
DOC_WEIGHT = np.array([min(RARE_BOOST, math.sqrt(_mx / max(1.0, TRAIN_LABEL_COUNTS[d["y"] > 0].min())))
                       if d["y"].any() else 1.0 for d in DATA["train"]])

def epoch_sample(docs, rng):
    n = min(TRAIN_DOCS_PER_EPOCH or len(docs), len(docs))
    w = DOC_WEIGHT / DOC_WEIGHT.sum()
    idx = np.random.default_rng(rng.randrange(1 << 30)).choice(len(docs), size=n, replace=False, p=w)
    return [docs[i] for i in idx]

def collate_docs(batch):
    chunks = [c for it in batch for c in it["chunks"]]
    W = len(chunks)
    T = max(len(c["ids"]) for c in chunks)
    M = (XL_MEMORY + XL_MEMORY_OLD) if any(len(c["mem"]) for c in chunks) else 0
    ids = torch.full((W, T), PAD_ID, dtype=torch.long); mask = torch.zeros((W, T), dtype=torch.bool)
    mem_ids = torch.full((W, M), PAD_ID, dtype=torch.long); mem_mask = torch.zeros((W, M), dtype=torch.bool)
    starts = torch.zeros(W, dtype=torch.float32)
    c_doc, c_slot, c_bucket = [], [], []
    wi = 0
    for di, it in enumerate(batch):
        for si, c in enumerate(it["chunks"]):
            n = len(c["ids"])
            ids[wi, :n] = torch.from_numpy(c["ids"]); mask[wi, :n] = True
            k = len(c["mem"])
            if k:
                mem_ids[wi, M - k:] = torch.from_numpy(c["mem"]); mem_mask[wi, M - k:] = True
            starts[wi] = c["start"]
            c_doc.append(di); c_slot.append(si); c_bucket.append(c["bucket"])
            wi += 1
    t = lambda x: torch.tensor(x, dtype=torch.long)
    out = dict(ids=ids, mask=mask, mem_ids=mem_ids, mem_mask=mem_mask, starts=starts,
               c_doc=t(c_doc), c_slot=t(c_slot), c_bucket=t(c_bucket),
               n_chunks=t([len(it["chunks"]) for it in batch]))
    if all("y" in it for it in batch):
        out["labels"] = torch.from_numpy(np.stack([it["y"] for it in batch]))
    return out

def to_device(batch):
    return {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

_demo = make_doc_item(DATA["dev"][0], SELECTION, N_CHUNKS)
print(f"example: {DATA['dev'][0]['n_tokens']:,} tokens -> {len(_demo['chunks'])} chunks starting at "
      f"{[c['start'] for c in _demo['chunks']]} (position buckets {[c['bucket'] for c in _demo['chunks']]})")

In [ ]:
def sinusoidal_relative_positions(max_distance, d_model, device, base=10_000.0):
    positions = torch.arange(max_distance + 1, device=device, dtype=torch.float32)
    inv_freq = torch.exp(-math.log(base) * torch.arange(0, d_model, 2, device=device, dtype=torch.float32) / d_model)
    angles = positions[:, None] * inv_freq[None, :]
    return torch.cat((angles.sin(), angles.cos()), dim=-1)

class RelativeMultiheadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout, rel_base=10_000.0):
        super().__init__()
        self.d_model, self.n_heads = d_model, n_heads
        self.head_dim = d_model // n_heads
        self.rel_base = rel_base
        self.attn_dropout = nn.Dropout(dropout)
        self.q_net = nn.Linear(d_model, d_model, bias=False)
        self.k_net = nn.Linear(d_model, d_model, bias=False)
        self.v_net = nn.Linear(d_model, d_model, bias=False)
        self.r_net = nn.Linear(d_model, d_model, bias=False)
        self.o_net = nn.Linear(d_model, d_model, bias=False)
        self.r_w_bias = nn.Parameter(torch.zeros(n_heads, self.head_dim))
        self.r_r_bias = nn.Parameter(torch.zeros(n_heads, self.head_dim))

    def _split_heads(self, x):
        b, t, _ = x.shape
        return x.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(self, x, memory=None, key_mask=None):
        b, qlen, _ = x.shape
        mlen = 0 if memory is None else memory.size(1)
        kv_input = x if memory is None else torch.cat((memory.to(x.dtype), x), dim=1)
        klen = kv_input.size(1)
        q = self._split_heads(self.q_net(x))
        k = self._split_heads(self.k_net(kv_input))
        value = self._split_heads(self.v_net(kv_input))
        ac = torch.matmul((q + self.r_w_bias[None, :, None, :]).float(), k.float().transpose(-1, -2))
        rel_pos = sinusoidal_relative_positions(mlen + qlen - 1, self.d_model, x.device, self.rel_base)
        rel_keys = self._split_heads(self.r_net(rel_pos[None].to(self.r_net.weight.dtype)))[0].transpose(0, 1)
        bd_by_distance = torch.einsum("bhqd,rhd->bhqr", (q + self.r_r_bias[None, :, None, :]).float(), rel_keys.float())
        key_pos = torch.arange(-mlen, qlen, device=x.device)
        query_pos = torch.arange(qlen, device=x.device)
        rel_dist = (query_pos[:, None] - key_pos[None, :]).clamp_min(0)
        bd = torch.gather(bd_by_distance, dim=-1, index=rel_dist[None, None].expand(b, self.n_heads, qlen, klen))
        scores = (ac + bd) * (self.head_dim ** -0.5)
        allowed = (key_pos[None, :] <= query_pos[:, None])[None, None]
        if key_mask is not None:
            allowed = allowed & key_mask[:, None, None, :]
        scores = scores.masked_fill(~allowed, torch.finfo(scores.dtype).min)
        weights = self.attn_dropout(torch.softmax(scores, dim=-1).to(value.dtype))
        out = torch.matmul(weights, value).transpose(1, 2).contiguous().view(b, qlen, self.d_model)
        return self.o_net(out)

class FeedForward(nn.Module):
    def __init__(self, d_model, hidden, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, hidden), nn.GELU(), nn.Linear(hidden, d_model), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)

class TransformerXLBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_hidden, dropout, rel_base):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = RelativeMultiheadAttention(d_model, n_heads, dropout, rel_base)
        self.ff = FeedForward(d_model, ff_hidden, dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, memory=None, key_mask=None):
        x = x + self.resid_dropout(self.attn(self.norm1(x), memory, key_mask))
        return x + self.resid_dropout(self.ff(self.norm2(x)))

class TXLBackbone(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, ff_hidden, dropout, rel_base=10_000.0):
        super().__init__()
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.embedding_dropout = nn.Dropout(dropout)
        self.layers = nn.ModuleList([TransformerXLBlock(d_model, n_heads, ff_hidden, dropout, rel_base)
                                     for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)

    @property
    def blocks(self):
        return self.layers

    def forward(self, input_ids, mems=None, key_mask=None, return_hidden=False, grad_ckpt=False):
        h = self.embedding_dropout(self.token_embedding(input_ids))
        levels = [h]
        for i, layer in enumerate(self.layers):
            mem = None if mems is None else mems[i]
            if grad_ckpt:
                h = checkpoint(layer, h, mem, key_mask, use_reentrant=False)
            else:
                h = layer(h, mem, key_mask)
            levels.append(h)
        out = self.final_norm(h)
        return (out, levels) if return_hidden else out

def build_backbone(n_layers=None, layer_map=None, dropout=None):
    c = P1_CFG
    n = n_layers or c["layers"]
    bb = TXLBackbone(VOCAB_SIZE, c["d_model"], c["heads"], n, c["ff_hidden"],
                     BACKBONE_DROPOUT if dropout is None else dropout, c.get("rel_base", 10_000.0))
    sd = {k: v for k, v in P1_STATE.items() if not k.startswith("lm_head")}
    if layer_map is not None:
        assert len(layer_map) == n
        new_sd = {k: v for k, v in sd.items() if not k.startswith("layers.")}
        for new_i, old_i in enumerate(layer_map):
            pre = f"layers.{old_i}."
            for k, v in sd.items():
                if k.startswith(pre):
                    new_sd[f"layers.{new_i}." + k[len(pre):]] = v
        sd = new_sd
    missing, unexpected = bb.load_state_dict(sd, strict=False)
    if missing or unexpected:
        raise RuntimeError(f"Phase-1 checkpoint does not match backbone. missing={missing[:5]} unexpected={unexpected[:5]}")
    return bb

def build_student_backbone(cfg, drop):
    n = cfg.get("student_layers") or P1_CFG["layers"]
    lm = [int(round(x)) for x in np.linspace(0, P1_CFG["layers"] - 1, n)]
    return build_backbone(n_layers=n, layer_map=lm, dropout=drop), lm

class TokenEncoder(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone, self.out_dim = backbone, backbone.d_model

    def forward(self, ids, mask, mem_ids=None, mem_mask=None, starts=None, return_hidden=False, grad_ckpt=False):
        bb = self.backbone
        mems, key_mask = None, mask
        if mem_ids is not None and mem_ids.size(1) > 0:
            with torch.no_grad():
                _, lv = bb(mem_ids, key_mask=mem_mask, return_hidden=True)
            mems = [x.detach() for x in lv[:-1]]
            key_mask = torch.cat([mem_mask, mask], 1)
        out = bb(ids, mems=mems, key_mask=key_mask, return_hidden=return_hidden, grad_ckpt=grad_ckpt)
        if return_hidden:
            return out[0], out[1][1:]
        return out

_probe = build_backbone()
P1_PARAMS = sum(p.numel() for p in _probe.parameters())
print(f"Phase-1 Transformer-XL: {P1_PARAMS/1e6:.2f}M parameters (tied embedding counted once)")
assert MIN_PARAMS <= P1_PARAMS <= MAX_PARAMS or SMOKE_TEST
del _probe

In [ ]:
class LoRMAPlusLinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)
        y = y0 + self.scale * F.linear(F.linear(y0, self.A), self.B)
        return y if self.base.bias is None else y + self.base.bias

NF4_LEVELS = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635,
                           -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
                           0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
                           0.7229568362236023, 1.0])
QBLOCK = 64

def nf4_quantize(w):
    flat = w.detach().float().reshape(-1)
    pad = (-flat.numel()) % QBLOCK
    flat = torch.cat([flat, flat.new_zeros(pad)])
    blocks = flat.view(-1, QBLOCK)
    absmax = blocks.abs().max(1).values.clamp(min=1e-8)
    codes = (blocks / absmax[:, None]).unsqueeze(-1).sub(NF4_LEVELS).abs().argmin(-1).to(torch.uint8).view(-1)
    packed = (codes[0::2] | (codes[1::2] << 4)).contiguous()
    return packed, absmax.half(), pad

class QLoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.out_features, self.in_features = base.out_features, base.in_features
        packed, absmax, pad = nf4_quantize(base.weight)
        self.register_buffer("qweight", packed)
        self.register_buffer("absmax", absmax)
        self.register_buffer("levels", NF4_LEVELS.clone())
        self.pad = pad
        self.bias = None if base.bias is None else nn.Parameter(base.bias.detach().clone(), requires_grad=False)
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, self.in_features))
        self.B = nn.Parameter(torch.zeros(self.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def dequantize(self, dtype):
        codes = torch.stack([self.qweight & 15, self.qweight >> 4], 1).view(-1).long()
        w = (self.levels[codes].view(-1, QBLOCK) * self.absmax.float()[:, None]).view(-1)
        if self.pad:
            w = w[:-self.pad]
        return w.view(self.out_features, self.in_features).to(dtype)

    def forward(self, x):

        y = F.linear(x, self.dequantize(torch.float32), self.bias)
        return y + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)

class AdaLoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r = base, r
        self.scale = alpha / r
        self.Q = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.E = nn.Parameter(torch.zeros(r))
        self.P = nn.Parameter(torch.randn(base.out_features, r) * 0.02)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.Q) * self.E.to(x.dtype), self.P)

    def orth_penalty(self):
        I = torch.eye(self.r, device=self.Q.device)
        return (torch.norm(self.Q @ self.Q.T - I, p="fro") + torch.norm(self.P.T @ self.P - I, p="fro"))

class RankAllocator:
    def __init__(self, modules, cfg, total_steps):
        self.mods, self.cfg, self.total = modules, cfg, total_steps
        self.b1, self.b2 = cfg["beta1"], cfg["beta2"]
        self.init_budget = sum(m.r for m in modules)
        self.target_budget = cfg["target_rank"] * len(modules)
        self.ipt, self.unc = {}, {}
        self.final_keep = None
        self.budget = self.init_budget

    @torch.no_grad()
    def update_importance(self):
        for mi, m in enumerate(self.mods):
            for name in ("P", "E", "Q"):
                p = getattr(m, name)
                if p.grad is None:
                    continue
                cur = (p * p.grad).abs().float()
                key = (mi, name)
                if key not in self.ipt:
                    self.ipt[key], self.unc[key] = torch.zeros_like(cur), torch.zeros_like(cur)
                self.ipt[key].mul_(self.b1).add_((1 - self.b1) * cur)
                self.unc[key].mul_(self.b2).add_((1 - self.b2) * (cur - self.ipt[key]).abs())

    def _scores(self):
        out = []
        for mi, m in enumerate(self.mods):
            s = lambda n: self.ipt.get((mi, n), torch.zeros(1, device=m.E.device)) * \
                self.unc.get((mi, n), torch.zeros(1, device=m.E.device))
            sE = s("E").reshape(-1) if (mi, "E") in self.ipt else torch.zeros(m.r, device=m.E.device)
            sP = s("P").mean(0) if (mi, "P") in self.ipt else torch.zeros(m.r, device=m.E.device)
            sQ = s("Q").mean(1) if (mi, "Q") in self.ipt else torch.zeros(m.r, device=m.E.device)
            out.append(sE + sP + sQ)
        return out

    @torch.no_grad()
    def _apply(self, keep):
        for m, k in zip(self.mods, keep):
            m.E.masked_fill_(~k, 0.0)

    @torch.no_grad()
    def step(self, step):
        c, T = self.cfg, self.total
        t_init, t_final = int(c["t_init"] * T), int(c["t_final"] * T)
        if step <= t_init:
            return
        if step > T - t_final:
            if self.final_keep is None:
                self.final_keep = self._keep(self.target_budget)
            self._apply(self.final_keep)
            self.budget = self.target_budget
            return
        frac = 1 - (step - t_init) / max(1, T - t_final - t_init)
        self.budget = int((self.init_budget - self.target_budget) * frac ** 3 + self.target_budget)
        if step % c["delta_t"] == 0:
            self._apply(self._keep(self.budget))

    def _keep(self, budget):
        scores = self._scores()
        flat = torch.cat(scores)
        if budget >= flat.numel():
            return [torch.ones_like(s, dtype=torch.bool) for s in scores]
        keep = torch.zeros(flat.numel(), dtype=torch.bool, device=flat.device)
        keep[torch.topk(flat, budget).indices] = True
        return list(torch.split(keep, [s.numel() for s in scores]))

    def ranks(self):
        return [int((m.E != 0).sum()) for m in self.mods]

class HoulsbyAdapter(nn.Module):
    def __init__(self, base, bottleneck):
        super().__init__()
        self.base = base
        d = base.out_features
        self.down = nn.Linear(d, bottleneck)
        self.up = nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        y = self.base(x)
        return y + self.up(F.gelu(self.down(y)))

ADAPTED_TYPES = (LoRMAPlusLinear, QLoRALinear, AdaLoRALinear, HoulsbyAdapter)

def _resolve(block, path):
    parent = block
    parts = path.split(".")
    for p in parts[:-1]:
        parent = getattr(parent, p)
    return parent, parts[-1], getattr(parent, parts[-1])

def wrap_linears(backbone, targets, factory):
    n = 0
    for block in backbone.blocks:
        for path in targets:
            try:
                parent, name, leaf = _resolve(block, path)
            except AttributeError:
                continue
            if isinstance(leaf, nn.Linear):
                setattr(parent, name, factory(leaf)); n += 1
    return n

def is_norm(m):
    return isinstance(m, nn.LayerNorm) or "RMSNorm" in type(m).__name__

def block_param_ids(bb):
    return [{id(p) for p in b.parameters()} for b in bb.blocks]

def top_extras(bb):
    inside = set().union(*block_param_ids(bb))
    return [p for n, p in bb.named_parameters() if id(p) not in inside and "emb" not in n.split(".")[0]]

def configure_trainable(model, cfg):
    method = cfg["method"]
    bb = model.encoder.backbone
    for p in model.parameters():
        p.requires_grad = False
    save = set()
    if method == "frozen":
        pass
    elif method in ("full", "distill"):
        for p in model.parameters():
            p.requires_grad = True
    elif method == "lorma_plus":
        for m in bb.modules():
            if isinstance(m, LoRMAPlusLinear):
                m.A.requires_grad = m.B.requires_grad = True
    elif method == "adalora":
        for m in bb.modules():
            if isinstance(m, AdaLoRALinear):
                m.P.requires_grad = m.E.requires_grad = m.Q.requires_grad = True
    elif method == "qlora":
        for m in bb.modules():
            if isinstance(m, QLoRALinear):
                m.A.requires_grad = m.B.requires_grad = True
    elif method == "prefix":
        for p in bb.prefix.parameters():
            p.requires_grad = True
    elif method == "adapter":
        for m in bb.modules():
            if isinstance(m, HoulsbyAdapter):
                for p in list(m.down.parameters()) + list(m.up.parameters()):
                    p.requires_grad = True
            elif is_norm(m):
                for p in m.parameters():
                    p.requires_grad = True
    elif method == "partial":
        for b in list(bb.blocks)[-cfg["top_layers"]:]:
            for p in b.parameters():
                p.requires_grad = True
        for p in top_extras(bb):
            p.requires_grad = True
    else:
        raise ValueError(method)
    for m in model.head_modules():
        for p in m.parameters():
            p.requires_grad = True
    save |= {n for n, p in model.named_parameters() if p.requires_grad}
    model.save_names = save

def param_groups(model, cfg):
    head_ids = {id(p) for m in model.head_modules() for p in m.parameters()}
    head = [p for p in model.parameters() if id(p) in head_ids]
    groups = [dict(params=head, lr=cfg["head_lr"], weight_decay=0.0)]
    decay = WEIGHT_DECAY if cfg["method"] in ("full", "distill", "partial") else 0.0
    bb = model.encoder.backbone
    if cfg["method"] == "partial":
        blocks = list(bb.blocks)[::-1]
        k = cfg["top_layers"]
        for depth, b in enumerate(blocks[:k]):
            groups.append(dict(params=list(b.parameters()), lr=cfg["lr"] * cfg["llrd"] ** depth, weight_decay=decay))
        groups.append(dict(params=top_extras(bb), lr=cfg["lr"], weight_decay=0.0))
    else:
        body = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
        if body:
            groups.append(dict(params=body, lr=cfg["lr"], weight_decay=decay))
    for g in groups:
        g["initial_lr"] = g["lr"]
    return [g for g in groups if g["params"]]

def make_controller(model, cfg, total_steps):
    if cfg["method"] == "adalora":
        mods = [m for m in model.encoder.backbone.modules() if isinstance(m, AdaLoRALinear)]
        return RankAllocator(mods, cfg, total_steps)
    return None

def extra_loss(model, cfg):
    if cfg["method"] != "adalora":
        return None
    mods = [m for m in model.encoder.backbone.modules() if isinstance(m, AdaLoRALinear)]
    return cfg["orth_reg"] * torch.stack([m.orth_penalty() for m in mods]).mean()

def build_doc_model(cfg):
    method = cfg["method"]
    drop = cfg["dropout"] if cfg.get("dropout") is not None else BACKBONE_DROPOUT
    if method == "distill":
        backbone, lm = build_student_backbone(cfg, drop)
        cfg["layer_map"] = lm
    else:
        backbone = build_backbone(dropout=drop)
    if method == "lorma_plus":
        n = wrap_linears(backbone, LOWRANK_TARGETS, lambda l: LoRMAPlusLinear(l, cfg["rank"], cfg["alpha"]))
        print(f"lorma_plus: adapted {n} linear layers")
    elif method == "adalora":
        n = wrap_linears(backbone, LOWRANK_TARGETS, lambda l: AdaLoRALinear(l, cfg["init_rank"], cfg["alpha"]))
        print(f"adalora: adapted {n} linear layers (rank {cfg['init_rank']} -> avg {cfg['target_rank']})")
    elif method == "adapter":
        n = wrap_linears(backbone, ADAPTER_TARGETS, lambda l: HoulsbyAdapter(l, cfg["bottleneck"]))
        print(f"adapter: {n} Houlsby adapters (bottleneck {cfg['bottleneck']})")
    elif method == "qlora":
        n = wrap_linears(backbone, LOWRANK_TARGETS, lambda l: QLoRALinear(l, cfg["rank"], cfg["alpha"]))
        print(f"qlora: {n} linear layers stored in 4-bit NF4 + LoRA rank {cfg['rank']}")
    elif method == "prefix":
        if not hasattr(backbone, "add_prefix"):
            raise ValueError("prefix tuning is implemented for the multi-resolution memory model only")
        backbone.add_prefix(cfg["prefix_len"], cfg["prefix_hidden"])
        print(f"prefix: {cfg['prefix_len']} learned prefix states in each of {len(backbone.blocks)} layers")
    cfg["selection"] = cfg.get("selection") or SELECTION
    cfg["n_chunks"] = cfg.get("n_chunks") or N_CHUNKS
    cfg["head"] = cfg.get("head") or HEAD
    cfg["loss"] = cfg.get("loss") or LOSS
    cfg["label_correlation"] = LABEL_CORRELATION if cfg.get("label_correlation") is None else cfg["label_correlation"]
    model = DocModel(TokenEncoder(backbone), cfg["head"], cfg["label_correlation"])
    model.cfg_selection = cfg["selection"]
    model.cfg_n_chunks = cfg.get("eval_n_chunks") or EVAL_N_CHUNKS or cfg["n_chunks"]
    model.grad_ckpt = USE_GRAD_CHECKPOINTING
    configure_trainable(model, cfg)
    return model

def _adapter_checks():
    torch.manual_seed(0)
    base = nn.Linear(24, 32)
    x = torch.randn(3, 24)
    ref = base(x)
    for name, m in [("lorma_plus", LoRMAPlusLinear(base, 4, 4)), ("adalora", AdaLoRALinear(base, 4, 8)),
                    ("houlsby", HoulsbyAdapter(base, 8))]:
        assert torch.allclose(m(x), ref, atol=1e-5), f"{name} not identity at init"
    q = QLoRALinear(base, 4, 8)
    rel = (q.dequantize(torch.float32) - base.weight).norm() / base.weight.norm()
    assert rel < 0.15, f"NF4 reconstruction error too high: {rel:.3f}"

    mods = [AdaLoRALinear(nn.Linear(16, 16), 6, 8) for _ in range(3)]
    for m in mods:
        m.E.data.uniform_(0.5, 1.0)
        for p in (m.P, m.E, m.Q):
            p.grad = torch.randn_like(p)
    ra = RankAllocator(mods, dict(beta1=0.85, beta2=0.85, target_rank=2, t_init=0.0, t_final=0.0, delta_t=1), 10)
    ra.update_importance()
    ra._apply(ra._keep(7))
    assert sum(ra.ranks()) == 7, ra.ranks()
    print(f"Adapter checks passed (LoRMA+, AdaLoRA, Houlsby = identity at init; AdaLoRA budget masking works; "
          f"NF4 4-bit relative weight error {rel:.3f}).")

_adapter_checks()

In [ ]:
class DocModel(nn.Module):
    def __init__(self, encoder, head, label_correlation):
        super().__init__()
        self.encoder, self.head_type = encoder, head
        d_in = encoder.out_dim * (2 if CHUNK_POOLING == "mean+last" else 1)
        self.chunk_proj = nn.Sequential(nn.Dropout(HEAD_DROPOUT), nn.Linear(d_in, HIDDEN), nn.GELU())
        self.pos_emb = nn.Embedding(N_POS_BUCKETS, HIDDEN)
        nn.init.normal_(self.pos_emb.weight, std=0.02)
        if head == "label_attention":

            self.label_q = nn.Parameter(torch.randn(NUM_LABELS, HIDDEN) * 0.02)
            self.label_w = nn.Parameter(torch.randn(NUM_LABELS, HIDDEN) * 0.02)
            self.label_b = nn.Parameter(torch.zeros(NUM_LABELS))
            self.att = self.classifier = None
        else:
            self.att = nn.Sequential(nn.Linear(HIDDEN, HIDDEN // 2), nn.Tanh(), nn.Linear(HIDDEN // 2, 1))
            self.classifier = nn.Sequential(nn.Dropout(HEAD_DROPOUT), nn.Linear(2 * HIDDEN, HIDDEN), nn.GELU(),
                                            nn.Dropout(HEAD_DROPOUT), nn.Linear(HIDDEN, NUM_LABELS))

        self.corr = nn.Linear(NUM_LABELS, NUM_LABELS) if label_correlation else None
        if self.corr is not None:
            nn.init.zeros_(self.corr.weight); nn.init.zeros_(self.corr.bias)
        self.grad_ckpt = False
        self.save_names = set()

    def head_modules(self):
        mods = [self.chunk_proj, self.pos_emb, self.att, self.classifier, self.corr]
        return [m for m in mods if m is not None] + ([self._label_params()] if self.head_type == "label_attention" else [])

    def _label_params(self):
        holder = nn.Module()
        holder.q, holder.w, holder.b = self.label_q, self.label_w, self.label_b
        return holder

    def chunk_vectors(self, b):
        h = self.encoder(b["ids"], b["mask"], b["mem_ids"], b["mem_mask"], b["starts"],
                         grad_ckpt=self.grad_ckpt and self.training).float()
        m = b["mask"].float()
        lens = m.sum(1).clamp(min=1)
        mean = (h * m[..., None]).sum(1) / lens[:, None]
        if CHUNK_POOLING == "mean+last":
            last = h[torch.arange(h.size(0), device=h.device), lens.long() - 1]
            return torch.cat([mean, last], -1)
        return mean

    def forward(self, b, return_aux=False):
        feats = self.chunk_vectors(b)
        x = self.chunk_proj(feats) + self.pos_emb(b["c_bucket"])
        B, K = b["n_chunks"].numel(), int(b["n_chunks"].max())
        grid = x.new_zeros(B, K, x.size(-1)); cmask = torch.zeros(B, K, dtype=torch.bool, device=x.device)
        grid[b["c_doc"], b["c_slot"]] = x; cmask[b["c_doc"], b["c_slot"]] = True
        if self.head_type == "label_attention":
            g = grid.float()
            s = torch.einsum("bkh,lh->blk", g, self.label_q.float()).masked_fill(~cmask[:, None, :], -1e4)
            w = torch.softmax(s, -1)
            v = torch.einsum("blk,bkh->blh", w, g)
            logits = (v * self.label_w.float()).sum(-1) + self.label_b.float()
            w_pos = w.mean(1)
        else:
            s = self.att(grid).squeeze(-1).masked_fill(~cmask, -1e4)
            w_pos = torch.softmax(s.float(), -1)
            main = (w_pos[..., None] * grid.float()).sum(1)
            mx = grid.float().masked_fill(~cmask[..., None], -1e4).max(1).values
            logits = self.classifier(torch.cat([main, mx], -1)).float()
        if self.corr is not None:
            logits = logits + self.corr(torch.sigmoid(logits))
        if return_aux:
            return logits, dict(feats=feats, weights=w_pos, cmask=cmask)
        return logits

@torch.no_grad()
def predict_docs(model, docs, mode=None, K=None, keep_weights=False):
    model.eval()
    mode, K = mode or model.cfg_selection, K or model.cfg_n_chunks
    bs = max(1, EVAL_BATCH_DOCS * N_CHUNKS // max(K, 1))
    logits, weights = [], {}
    t0 = time.time()
    for i in range(0, len(docs), bs):
        items = [make_doc_item(d, mode, K, with_label=False) for d in docs[i:i + bs]]
        b = to_device(collate_docs(items))
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
            lg, aux = model(b, return_aux=True)
        logits.append(lg.float().cpu())
        if keep_weights:
            w = aux["weights"].float().cpu().numpy()
            for j, it in enumerate(items):
                k = len(it["chunks"])
                weights[str(it["id"])] = ([c["bucket"] for c in it["chunks"]], w[j, :k].tolist())
    lg = torch.cat(logits) if logits else torch.zeros(0, NUM_LABELS)
    return dict(logits=lg.numpy(), prob=torch.sigmoid(lg).numpy(), weights=weights, seconds=time.time() - t0)

def multilabel_scores(Y, Pred):
    Y, Pred = np.asarray(Y, bool), np.asarray(Pred, bool)
    tp = (Y & Pred).sum(0).astype(float); fp = (~Y & Pred).sum(0).astype(float); fn = (Y & ~Pred).sum(0).astype(float)
    P = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    R = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    F = np.divide(2 * P * R, P + R, out=np.zeros_like(tp), where=(P + R) > 0)
    ref = Y.sum(0) > 0
    mtp, mfp, mfn = tp.sum(), fp.sum(), fn.sum()
    micro = 2 * mtp / max(1.0, 2 * mtp + mfp + mfn)
    return dict(macro_f1=float(F[ref].mean()) if ref.any() else 0.0, macro_precision=float(P[ref].mean()) if ref.any() else 0.0,
                macro_recall=float(R[ref].mean()) if ref.any() else 0.0, macro_f1_all_labels=float(F.mean()),
                micro_f1=float(micro), micro_precision=float(mtp / max(1.0, mtp + mfp)),
                micro_recall=float(mtp / max(1.0, mtp + mfn)), subset_accuracy=float((Y == Pred).all(1).mean()),
                avg_predicted=float(Pred.sum(1).mean()), per_label_f1=F.tolist(), per_label_p=P.tolist(),
                per_label_r=R.tolist(), support=Y.sum(0).astype(int).tolist())

def decide(prob, thr):
    pred = prob >= np.asarray(thr)[None] if np.ndim(thr) else prob >= thr
    if AT_LEAST_ONE and len(prob):
        empty = ~pred.any(1)
        pred[np.where(empty)[0], prob[empty].argmax(1)] = True
    return pred

THR_GRID = np.round(np.arange(0.02, 0.951, 0.01), 2)

def tune_global(Y, prob):
    best = (-1.0, 0.5)
    for t in THR_GRID[::2]:
        f1 = multilabel_scores(Y, decide(prob, t))["macro_f1"]
        if f1 > best[0] + 1e-9:
            best = (f1, float(t))
    return best[1]

def tune_per_label(Y, prob, base):
    Y = np.asarray(Y, bool)
    thr = np.full(Y.shape[1], base, dtype=np.float64)
    for l in np.where(Y.sum(0) >= MIN_DEV_SUPPORT_PER_LABEL)[0]:
        y, p = Y[:, l], prob[:, l]
        best = (-1.0, base)
        for t in THR_GRID:
            pr = p >= t
            tp = (pr & y).sum(); fp = (pr & ~y).sum(); fn = (~pr & y).sum()
            f1 = 2 * tp / max(1, 2 * tp + fp + fn)
            if f1 > best[0] + 1e-9 or (abs(f1 - best[0]) <= 1e-9 and abs(t - base) < abs(best[1] - base)):
                best = (f1, float(t))
        thr[l] = best[1]
    return thr

def threshold_study(Y, prob, seed=0):
    Y = np.asarray(Y)
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(Y)); halves = [idx[: len(Y) // 2], idx[len(Y) // 2:]]
    g = tune_global(Y, prob)
    pl = tune_per_label(Y, prob, g)
    k = max(1, int(round(Y_TRAIN.sum(1).mean())))
    topk = np.zeros_like(prob, dtype=bool)
    if len(prob):
        topk[np.arange(len(prob))[:, None], np.argsort(-prob, 1)[:, :k]] = True
    out = {"fixed_0.5": dict(thr=0.5, **multilabel_scores(Y, decide(prob, 0.5))),
           "global": dict(thr=g, **multilabel_scores(Y, decide(prob, g))),
           "per_label": dict(thr=pl.tolist(), **multilabel_scores(Y, decide(prob, pl))),
           f"top_{k}": dict(thr=None, **multilabel_scores(Y, topk))}
    for name in ("global", "per_label"):
        cv = []
        for a, b in ((0, 1), (1, 0)):
            ta, tb = halves[a], halves[b]
            tg = tune_global(Y[ta], prob[ta])
            t = tg if name == "global" else tune_per_label(Y[ta], prob[ta], tg)
            cv.append(multilabel_scores(Y[tb], decide(prob[tb], t))["macro_f1"])
        out[name]["cv_macro_f1"] = float(np.mean(cv))
    best = max(("global", "per_label"), key=lambda n: out[n]["cv_macro_f1"])
    return out, best

def fit_temperature(logits, Y):
    lg, y = torch.tensor(logits, dtype=torch.float64), torch.tensor(np.asarray(Y), dtype=torch.float64)
    grid = np.exp(np.linspace(np.log(0.25), np.log(4.0), 80))
    nll = [float(F.binary_cross_entropy_with_logits(lg / T, y)) for T in grid]
    return float(grid[int(np.argmin(nll))])

def ece_multilabel(prob, Y, bins=10):
    p, y = np.asarray(prob).ravel(), np.asarray(Y).ravel()
    e, edges = 0.0, np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        sel = (p >= lo) & ((p < hi) | (hi == 1.0))
        if sel.any():
            e += sel.mean() * abs(y[sel].mean() - p[sel].mean())
    return float(e)

In [ ]:
TRAINED_MODELS, RESULTS = {}, {}

def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if (p.requires_grad or not trainable_only))

def logical_params(module):
    q = sum(m.out_features * m.in_features for m in module.modules() if isinstance(m, QLoRALinear))
    return count_params(module) + q

def make_cfg(name):
    cfg = dict(EXPERIMENT_DEFAULTS)
    cfg.update(copy.deepcopy(EXPERIMENTS[name]))
    cfg["name"] = name
    cfg["epochs"] = cfg.get("epochs") or EPOCHS
    return cfg

POS_WEIGHT = torch.tensor(np.clip(np.sqrt((len(Y_TRAIN) - TRAIN_LABEL_COUNTS) / np.maximum(TRAIN_LABEL_COUNTS, 1)),
                                  1.0, POS_WEIGHT_CAP), dtype=torch.float32)

def multilabel_loss(logits, y, kind):
    z, y = logits.float(), y.float()
    if kind == "bce":
        l = F.binary_cross_entropy_with_logits(z, y, reduction="none")
    elif kind == "bce_pos_weight":
        l = F.binary_cross_entropy_with_logits(z, y, pos_weight=POS_WEIGHT.to(z.device), reduction="none")
    elif kind == "focal":
        p = torch.sigmoid(z)
        ce = F.binary_cross_entropy_with_logits(z, y, reduction="none")
        pt = p * y + (1 - p) * (1 - y)
        l = ce * (1 - pt) ** 2
    elif kind == "asl":
        p = torch.sigmoid(z)
        pn = (p - ASL_CLIP).clamp(min=0)
        lp = y * torch.log(p.clamp(min=1e-8)) * (1 - p) ** ASL_GAMMA_POS
        ln = (1 - y) * torch.log((1 - pn).clamp(min=1e-8)) * pn ** ASL_GAMMA_NEG
        l = -(lp + ln)
    else:
        raise ValueError(kind)
    return l.sum(1).mean()

def saved_state(model):
    return {k: v.detach().cpu() for k, v in model.state_dict().items() if k in model.save_names}

def load_trained(name):
    run_dir = OUTPUT_DIR / "runs" / name
    cfg = json.loads((run_dir / "cfg.json").read_text())
    model = build_doc_model(cfg)
    state = torch.load(run_dir / "best_state.pt", map_location="cpu")
    missing = {n for n, p in model.named_parameters() if p.requires_grad} - set(state)
    assert not missing, f"{name}: checkpoint misses {sorted(missing)[:3]}"
    model.load_state_dict(state, strict=False)
    model.thresholds = np.asarray(RESULTS.get(name, {}).get("thresholds", 0.5))
    return model

def write_submission(run_dir, prob, thr):
    pred = decide(prob, thr)
    sub = [{"id": d["id"], "pred": sorted({int(LABEL_IDS[j]) for j in np.where(row)[0]})}
           for d, row in zip(DATA["test"], pred)]
    valid = set(int(x) for x in LABEL_IDS)
    assert [s["id"] for s in sub] == test_ids
    assert all(len(s["pred"]) == len(set(s["pred"])) and set(s["pred"]) <= valid for s in sub)
    pj = Path(run_dir) / "prediction.json"
    pj.write_text(json.dumps(sub, ensure_ascii=False), encoding="utf-8")
    zp = Path(run_dir) / "submission.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    return zp

def kd_loss(s_logits, s_feats, t_logits, t_feats, y, cfg):
    T = cfg["kd_temperature"]
    target = torch.stack([torch.sigmoid(t.float() / T) for t in t_logits]).mean(0)
    kd = F.binary_cross_entropy_with_logits(s_logits.float() / T, target, reduction="none").sum(1).mean() * T * T
    hard = multilabel_loss(s_logits, y, cfg["loss"])
    tf = torch.stack(t_feats).mean(0)
    hid = F.mse_loss(F.layer_norm(s_feats.float(), (s_feats.size(-1),)), F.layer_norm(tf.float(), (tf.size(-1),)))
    return (1 - cfg["kd_alpha"]) * kd + cfg["kd_alpha"] * hard + cfg["kd_hidden_weight"] * hid

_dev_rng = random.Random(SEED)
DEV_SEL = sorted(_dev_rng.sample(range(len(DATA["dev"])), min(EPOCH_DEV_DOCS or len(DATA["dev"]), len(DATA["dev"]))))

def dev_f1(model):
    docs = [DATA["dev"][i] for i in DEV_SEL]
    ev = predict_docs(model, docs)
    Y = Y_DEV[DEV_SEL]
    g = tune_global(Y, ev["prob"])
    return multilabel_scores(Y, decide(ev["prob"], g))["macro_f1"], multilabel_scores(Y, decide(ev["prob"], 0.5))["macro_f1"], g

def run_experiment(name, teacher=None):
    cfg = make_cfg(name)
    run_dir = OUTPUT_DIR / "runs" / name
    run_dir.mkdir(parents=True, exist_ok=True)
    if RESUME_COMPLETED_RUNS and (run_dir / "results.json").exists():
        RESULTS[name] = json.loads((run_dir / "results.json").read_text())
        print(f"[{name}] already finished (dev macro-F1 {RESULTS[name]['dev']['macro_f1']:.4f}) - skipping")
        return RESULTS[name]
    print("=" * 100); print(f"{name}: {cfg['description']}"); print("=" * 100)
    seed_everything(SEED)
    is_kd = cfg["method"] == "distill"
    model = build_doc_model(cfg).to(DEVICE)
    n_bb, n_total = logical_params(model.encoder), logical_params(model)
    n_train = sum(p.numel() for n, p in model.named_parameters() if n in model.save_names)
    print(f"backbone (+adapters) {n_bb/1e6:.2f}M | whole model {n_total/1e6:.2f}M | trainable {n_train/1e6:.3f}M "
          f"({100*n_train/n_total:.2f}%) | {cfg['n_chunks']} x {CHUNK_TOKENS} tokens, selection={cfg['selection']}, "
          f"head={cfg['head']}, loss={cfg['loss']}, label correlation={cfg['label_correlation']}")
    assert MIN_PARAMS <= n_bb <= MAX_PARAMS or SMOKE_TEST, \
        f"backbone {n_bb/1e6:.1f}M outside the {MIN_PARAMS/1e6:.0f}M-{MAX_PARAMS/1e6:.0f}M limit"
    teachers = [] if teacher is None else (list(teacher) if isinstance(teacher, (list, tuple)) else [teacher])
    if is_kd:
        assert teachers, "distillation needs a teacher"
        for t in teachers:
            t.to(DEVICE).eval()
            for p in t.parameters():
                p.requires_grad = False
        print(f"teacher: {sum(logical_params(t.encoder) for t in teachers)/1e6:.1f}M backbone parameters "
              f"-> student {n_bb/1e6:.1f}M (layers {cfg['layer_map']})")
    groups = param_groups(model, cfg)
    optim = torch.optim.AdamW(groups, betas=(0.9, 0.98), eps=1e-8)
    accum = GRAD_ACCUM
    n_epoch_docs = min(TRAIN_DOCS_PER_EPOCH or len(DATA["train"]), len(DATA["train"]))
    steps_per_epoch = max(1, math.ceil(n_epoch_docs / (BATCH_DOCS * accum)))
    total = steps_per_epoch * cfg["epochs"]
    warm = max(1, int(WARMUP_FRACTION * total))
    sched = torch.optim.lr_scheduler.LambdaLR(optim, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
    controller = make_controller(model, cfg, total)
    use_amp = cfg["use_amp"] and DEVICE.type == "cuda"
    scaler = make_grad_scaler(use_amp)
    params = [p for g in groups for p in g["params"]]
    print(f"{n_epoch_docs:,} documents/epoch x {cfg['epochs']} epochs | optimizer steps {total} | "
          f"time budget {MAX_TRAIN_MINUTES} min")
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    rng = random.Random(SEED)
    history, best, best_ep, bad, t0, gstep, out_of_time = [], -1.0, 0, 0, time.time(), 0, False

    def step():
        nonlocal gstep
        if scaler is not None:
            scaler.unscale_(optim)
        if controller is not None:
            controller.update_importance()
        torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
        if scaler is None:
            optim.step()
        else:
            scaler.step(optim); scaler.update()
        sched.step(); optim.zero_grad(set_to_none=True)
        gstep += 1
        if controller is not None:
            controller.step(gstep)

    for ep in range(1, cfg["epochs"] + 1):
        docs = epoch_sample(DATA["train"], rng)
        loader = DataLoader(DocDataset(docs, cfg["selection"], cfg["n_chunks"]), batch_size=BATCH_DOCS, shuffle=False,
                            collate_fn=collate_docs, num_workers=2 if DEVICE.type == "cuda" else 0)
        model.train()
        run_loss, n_loss, micro, te, n_bad = 0.0, 0, 0, time.time(), 0
        optim.zero_grad(set_to_none=True)
        for b in tqdm(loader, desc=f"{name} ep{ep}", leave=False):
            b = to_device(b)
            try:
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                    if is_kd:
                        lg, aux = model(b, return_aux=True)
                        with torch.no_grad():
                            t_out = [t(b, return_aux=True) for t in teachers]
                        loss = kd_loss(lg, aux["feats"], [o[0] for o in t_out], [o[1]["feats"] for o in t_out],
                                       b["labels"], cfg)
                    else:
                        loss = multilabel_loss(model(b), b["labels"], cfg["loss"])
                    reg = extra_loss(model, cfg)
                    if reg is not None:
                        loss = loss + reg
                if not torch.isfinite(loss):
                    n_bad += 1
                    optim.zero_grad(set_to_none=True); continue
                (loss / accum).backward() if scaler is None else scaler.scale(loss / accum).backward()
            except torch.cuda.OutOfMemoryError:
                optim.zero_grad(set_to_none=True); torch.cuda.empty_cache()
                if not model.grad_ckpt:
                    model.grad_ckpt = True
                    print("    CUDA out of memory -> gradient checkpointing switched on")
                continue
            micro += 1; run_loss += float(loss); n_loss += 1
            if micro % accum == 0:
                step()
            if micro == 20 * accum and ep == 1:
                eta = (time.time() - te) / micro * len(loader) / 60
                print(f"    ~{eta:.0f} min per epoch at the current speed")
            if time.time() - t0 > MAX_TRAIN_MINUTES * 60:
                out_of_time = True
                print(f"    time budget of {MAX_TRAIN_MINUTES} min reached in epoch {ep} ({micro}/{len(loader)} batches)")
                break
        if micro % accum:
            step()
        if n_bad:
            print(f"    {n_bad} batch(es) skipped because of a non-finite loss")
        f1, f1_05, thr = dev_f1(model)
        history.append(dict(epoch=ep, train_loss=run_loss / max(1, n_loss), dev_macro_f1=f1, dev_macro_f1_at_05=f1_05,
                            threshold=thr, elapsed_s=time.time() - t0,
                            adalora_budget=getattr(controller, "budget", None)))
        print(f"  epoch {ep} | loss {history[-1]['train_loss']:.4f} | dev macro-F1 {f1:.4f} (global threshold {thr:.2f}; "
              f"{f1_05:.4f} at 0.5; {len(DEV_SEL)} dev docs) | {(time.time()-t0)/60:.1f} min")
        if f1 > best + 1e-5:
            best, best_ep, bad = f1, ep, 0
            torch.save(saved_state(model), run_dir / "best_state.pt")
        else:
            bad += 1
            if bad >= cfg["patience"]:
                print(f"  early stop (no improvement for {cfg['patience']} epochs)")
                break
        if out_of_time:
            break
    train_time = time.time() - t0
    peak = torch.cuda.max_memory_allocated() / 1024**3 if DEVICE.type == "cuda" else 0.0
    model.load_state_dict(torch.load(run_dir / "best_state.pt", map_location=DEVICE), strict=False)

    ev = predict_docs(model, DATA["dev"], keep_weights=True)
    study, strategy = threshold_study(Y_DEV, ev["prob"])
    thr = study[strategy]["thr"]
    dev = {k: v for k, v in study[strategy].items() if k != "thr"}
    temp = fit_temperature(ev["logits"], Y_DEV)
    p_cal = torch.sigmoid(torch.tensor(ev["logits"]) / temp).numpy()
    print(f"[{name}] best epoch {best_ep} | thresholds: {strategy} (cross-tuned macro-F1 {study[strategy]['cv_macro_f1']:.4f}) | "
          f"dev macro-P/R/F1 = {dev['macro_precision']:.4f}/{dev['macro_recall']:.4f}/{dev['macro_f1']:.4f} | "
          f"micro-F1 {dev['micro_f1']:.4f} | subset acc {dev['subset_accuracy']:.4f} | "
          f"(0.5: {study['fixed_0.5']['macro_f1']:.4f}, global: {study['global']['macro_f1']:.4f}) | "
          f"{len(Y_DEV) / ev['seconds']:.1f} docs/s")
    np.savez_compressed(run_dir / "dev_outputs.npz", logits=ev["logits"].astype(np.float32))
    save_json(run_dir / "dev_chunk_weights.json", ev["weights"])
    out = dict(name=name, method=cfg["method"], description=cfg["description"], cfg=cfg,
               params_backbone=n_bb, params_total=n_total, params_trainable=n_train,
               trainable_pct=100 * n_train / n_total, train_time_s=train_time, best_epoch=best_ep,
               epochs_run=len(history), peak_mem_gb=peak, threshold_strategy=strategy, thresholds=thr,
               threshold_study={k: {kk: vv for kk, vv in v.items() if not kk.startswith("per_label") and kk != "support"}
                                for k, v in study.items()},
               temperature=temp, ece_raw=ece_multilabel(ev["prob"], Y_DEV), ece_calibrated=ece_multilabel(p_cal, Y_DEV),
               dev=dev, docs_per_s=len(Y_DEV) / max(ev["seconds"], 1e-9),
               state_mb=(run_dir / "best_state.pt").stat().st_size / 1024**2, history=history)
    if controller is not None and hasattr(controller, "ranks"):
        out["adalora_ranks"] = controller.ranks()
        print(f"        AdaLoRA ranks in the selected checkpoint: {sum(out['adalora_ranks'])} triplets over "
              f"{len(out['adalora_ranks'])} matrices (budget at the end of training: {getattr(controller, 'budget', None)})")
    if WRITE_SUBMISSIONS:
        tp = predict_docs(model, DATA["test"])
        np.savez_compressed(run_dir / "test_outputs.npz", logits=tp["logits"].astype(np.float32))
        out["submission_zip"] = str(write_submission(run_dir, tp["prob"], np.asarray(thr)))
        out["test_avg_statutes"] = float(decide(tp["prob"], np.asarray(thr)).sum(1).mean())
        print(f"        submission: {out['submission_zip']} ({out['test_avg_statutes']:.2f} statutes per test document)")
    save_json(run_dir / "cfg.json", cfg)
    save_json(run_dir / "results.json", out)
    RESULTS[name] = out
    model.thresholds = np.asarray(thr)
    model.cpu(); TRAINED_MODELS[name] = model
    for t in teachers:
        t.cpu()
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return out

In [ ]:
for _p in sorted((OUTPUT_DIR / "runs").glob("*/results.json")):
    if _p.parent.name in EXPERIMENTS:
        RESULTS[_p.parent.name] = json.loads(_p.read_text())

TEACHER_CANDIDATES = [n for n, c in EXPERIMENTS.items() if c["method"] != "distill"]

def pick_teacher():
    done = {n: RESULTS[n]["dev"]["macro_f1"] for n in TEACHER_CANDIDATES if n in RESULTS}
    if not done:
        raise RuntimeError("Distillation needs a finished teacher: run the other experiments first.")
    name = max(done, key=done.get)
    return name, TRAINED_MODELS.get(name) or load_trained(name)

for name in RUN_LIST:
    teacher = None
    if EXPERIMENTS[name]["method"] == "distill" and not (OUTPUT_DIR / "runs" / name / "results.json").exists():
        tname, teacher = pick_teacher()
        EXPERIMENTS[name]["teacher_name"] = tname
        print(f"Teacher for {name}: {tname} (dev macro-F1 {RESULTS[tname]['dev']['macro_f1']:.4f})")
    run_experiment(name, teacher)
    gc.collect()
print("\nFinished:", [n for n in RUN_LIST if n in RESULTS])

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
ORDER = [n for n in EXPERIMENTS if n in RESULTS]
rows = []
for n in ORDER:
    r = RESULTS[n]; d = r["dev"]; ts = r["threshold_study"]
    rows.append(dict(experiment=n, method=r["method"], head=r["cfg"]["head"], loss=r["cfg"]["loss"],
                     label_corr=r["cfg"]["label_correlation"], backbone_M=r["params_backbone"] / 1e6,
                     trainable_M=r["params_trainable"] / 1e6, trainable_pct=r["trainable_pct"],
                     dev_macro_F1=d["macro_f1"], dev_micro_F1=d["micro_f1"], subset_acc=d["subset_accuracy"],
                     macro_P=d["macro_precision"], macro_R=d["macro_recall"], thresholds=r["threshold_strategy"],
                     F1_at_0_5=ts["fixed_0.5"]["macro_f1"], F1_global=ts["global"]["macro_f1"],
                     F1_per_label=ts["per_label"]["macro_f1"], F1_per_label_cv=ts["per_label"]["cv_macro_f1"],
                     ECE_raw=r["ece_raw"], ECE_cal=r["ece_calibrated"], train_min=r["train_time_s"] / 60,
                     peak_mem_GB=r["peak_mem_gb"], docs_per_s=r["docs_per_s"], ckpt_MB=r["state_mb"]))
summary = pd.DataFrame(rows).set_index("experiment")
summary.to_csv(OUTPUT_DIR / f"phase5_{ARCH_NAME}_results.csv")
display(summary.round(4))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
summary[["F1_at_0_5", "F1_global", "F1_per_label", "dev_micro_F1"]].plot.bar(ax=ax[0])
ax[0].set_title("Dev Macro-F1 by threshold strategy (+ Micro-F1)"); ax[0].tick_params(axis="x", rotation=30)
for n in ORDER:
    h = RESULTS[n]["history"]
    ax[1].plot([x["epoch"] for x in h], [x["dev_macro_f1"] for x in h], marker="o", label=n)
ax[1].set_title("Dev Macro-F1 per epoch (dev subset, global threshold)"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "phase5_overview.png", dpi=130); plt.show()

best_name = max(ORDER, key=lambda n: RESULTS[n]["dev"]["macro_f1"])
best = RESULTS[best_name]
print(f"Best run: {best_name} (dev Macro-F1 {best['dev']['macro_f1']:.4f}, thresholds: {best['threshold_strategy']})")

print("\n--- Threshold strategies for the best run ---")
display(pd.DataFrame({k: {m: v.get(m) for m in ("macro_f1", "cv_macro_f1", "macro_precision", "macro_recall",
                                                    "micro_f1", "subset_accuracy", "avg_predicted")}
                      for k, v in best["threshold_study"].items()}).T.round(4))
print(f"(gold statutes per dev document: {Y_DEV.sum(1).mean():.2f})")

print("\n--- Rare statutes: mean dev F1 by training-frequency tercile (statutes present in dev) ---")
in_dev = Y_DEV.sum(0) > 0
display(pd.DataFrame({n: {bk: float(np.mean(np.array(RESULTS[n]["dev"]["per_label_f1"])[(FREQ_BUCKET == bk) & in_dev]))
                          if ((FREQ_BUCKET == bk) & in_dev).any() else np.nan for bk in ("rare", "medium", "frequent")}
                      for n in ORDER}).round(4))

z = np.load(OUTPUT_DIR / "runs" / best_name / "dev_outputs.npz")
prob = 1 / (1 + np.exp(-z["logits"]))
pred = decide(prob, np.asarray(best["thresholds"]))
per = pd.DataFrame(dict(statute=LABEL_IDS, train_docs=TRAIN_LABEL_COUNTS.astype(int), dev_support=Y_DEV.sum(0).astype(int),
                        precision=best["dev"]["per_label_p"], recall=best["dev"]["per_label_r"],
                        f1=best["dev"]["per_label_f1"], bucket=FREQ_BUCKET))
per.to_csv(OUTPUT_DIR / "phase5_per_statute.csv", index=False)
print("\nMost often missed statutes (lowest recall, dev support >= 3):")
display(per[per.dev_support >= 3].sort_values(["recall", "dev_support"], ascending=[True, False]).head(10).round(3))
print(f"Correlation of per-statute F1 with log(training frequency): "
      f"{np.corrcoef(np.log1p(per.train_docs[in_dev]), per.f1[in_dev])[0, 1]:.3f}")

Yb = Y_DEV.astype(bool)
conf = Counter()
for i in range(len(Yb)):
    fn_, fp_ = np.where(Yb[i] & ~pred[i])[0], np.where(~Yb[i] & pred[i])[0]
    for a in fn_:
        for b in fp_:
            conf[(LABEL_IDS[a], LABEL_IDS[b])] += 1
print("\nMost frequent confusions in the same document (missed gold statute -> predicted instead):")
for (a, b), c in conf.most_common(10):
    print(f"   {a} -> {b}: {c}")

co = (Y_TRAIN.T @ Y_TRAIN); np.fill_diagonal(co, 0)
pairs = sorted(((co[i, j], i, j) for i in range(NUM_LABELS) for j in range(i + 1, NUM_LABELS) if co[i, j] > 0), reverse=True)[:10]
print("\n--- Statute co-occurrence (train) and how often the best run recovers BOTH when both are gold (dev) ---")
rows = []
for c, i, j in pairs:
    both = Yb[:, i] & Yb[:, j]
    rows.append(dict(pair=f"{LABEL_IDS[i]} + {LABEL_IDS[j]}", train_docs=int(c),
                     P_j_given_i=float(co[i, j] / max(1, TRAIN_LABEL_COUNTS[i])), dev_docs=int(both.sum()),
                     both_predicted=float((pred[both][:, [i, j]].all(1)).mean()) if both.any() else np.nan))
display(pd.DataFrame(rows).round(3))
k_gold = Y_DEV.sum(1)
doc_f1 = 2 * (Yb & pred).sum(1) / np.maximum(1, Yb.sum(1) + pred.sum(1))
print("Per-document F1 by number of gold statutes:")
display(pd.DataFrame(dict(k=np.minimum(k_gold, 4), f1=doc_f1, exact=(Yb == pred).all(1)))
        .groupby("k").agg(docs=("f1", "size"), doc_F1=("f1", "mean"), exact_match=("exact", "mean"))
        .rename(index={4: "4+"}).round(3))
corr_runs = [n for n in ORDER if RESULTS[n]["cfg"]["label_correlation"]]
if corr_runs:
    st = torch.load(OUTPUT_DIR / "runs" / corr_runs[0] / "best_state.pt", map_location="cpu")
    if "corr.weight" in st:
        C = st["corr.weight"].float().numpy()
        top = np.dstack(np.unravel_index(np.argsort(-np.abs(C).ravel())[:8], C.shape))[0]
        print(f"Strongest learned statute-to-statute links ({corr_runs[0]}): "
              + ", ".join(f"{LABEL_IDS[j]}->{LABEL_IDS[i]} ({C[i, j]:+.2f})" for i, j in top))

cw = json.loads((OUTPUT_DIR / "runs" / best_name / "dev_chunk_weights.json").read_text())
acc, cnt = np.zeros(N_POS_BUCKETS), np.zeros(N_POS_BUCKETS)
for buckets, ws in cw.values():
    if len(ws) > 1:
        for bk, w in zip(buckets, ws):
            acc[bk] += w * len(ws); cnt[bk] += 1
rel = pd.Series(np.where(cnt > 0, acc / np.maximum(cnt, 1), np.nan),
                index=[f"{i/N_POS_BUCKETS:.2f}-{(i+1)/N_POS_BUCKETS:.2f}" for i in range(N_POS_BUCKETS)])
print("\nMean attention given to a chunk by its relative position (multi-chunk documents; 1.0 = uniform):")
display(rel.round(3).to_frame("relative attention"))

SECTION_RESULTS = {}
if RUN_SECTION_ANALYSIS and best["method"] in EXPERIMENT_LIBRARY:
    model = (TRAINED_MODELS.get(best_name) or load_trained(best_name)).to(DEVICE)
    sel = sorted(random.Random(1).sample(range(len(DATA["dev"])), min(SECTION_ANALYSIS_DOCS, len(DATA["dev"]))))
    docs, Ys = [DATA["dev"][i] for i in sel], Y_DEV[sel]
    long_frac = np.mean([d["n_tokens"] > N_CHUNKS * CHUNK_TOKENS for d in docs])
    for mode, K in [("head", N_CHUNKS), ("middle", N_CHUNKS), ("tail", N_CHUNKS), ("head+tail", N_CHUNKS),
                    ("spread", N_CHUNKS), ("spread", 2 * N_CHUNKS), ("spread", 4 * N_CHUNKS)]:
        ev = predict_docs(model, docs, mode, K)
        g = tune_global(Ys, ev["prob"])
        SECTION_RESULTS[f"{mode} x{K}"] = dict(tokens=K * CHUNK_TOKENS,
                                               F1_trained_thresholds=multilabel_scores(Ys, decide(ev["prob"], np.asarray(best["thresholds"])))["macro_f1"],
                                               F1_global_retuned=multilabel_scores(Ys, decide(ev["prob"], g))["macro_f1"],
                                               micro_F1=multilabel_scores(Ys, decide(ev["prob"], g))["micro_f1"], seconds=ev["seconds"])
    sec = pd.DataFrame(SECTION_RESULTS).T
    print(f"\nSection comparison on {len(docs)} dev documents ({long_frac:.0%} longer than {N_CHUNKS} chunks) with "
          f"{best_name} (trained on {best['cfg']['selection']} x{best['cfg']['n_chunks']}):")
    display(sec.round(4))
    sec.to_csv(OUTPUT_DIR / "phase5_section_analysis.csv")
    model.cpu()

print("\n--- Key comparisons for the written analysis ---")
ref_name = best_name
ref = RESULTS[ref_name]["dev"]["macro_f1"]
for n in ORDER:
    r = RESULTS[n]
    if n != ref_name:
        print(f"{n:28s}: {r['dev']['macro_f1']:.4f} = {r['dev']['macro_f1']/max(ref, 1e-9):.1%} of {ref_name} "
              f"with {r['trainable_pct']:.2f}% trainable")
if "E4_distill" in RESULTS:
    r = RESULTS["E4_distill"]
    tn = r["cfg"].get("teacher_name")
    if tn in RESULTS:
        t = RESULTS[tn]
        print(f"KD: student {r['params_backbone']/1e6:.1f}M macro-F1 {r['dev']['macro_f1']:.4f} (micro {r['dev']['micro_f1']:.4f}) "
              f"vs teacher {tn} {t['params_backbone']/1e6:.1f}M {t['dev']['macro_f1']:.4f} (micro {t['dev']['micro_f1']:.4f}) | "
              f"speed {r['docs_per_s']:.1f} vs {t['docs_per_s']:.1f} docs/s | peak memory {r['peak_mem_gb']:.1f} vs {t['peak_mem_gb']:.1f} GB")

In [ ]:
def run_probs(name, split):
    z = np.load(OUTPUT_DIR / "runs" / name / f"{split}_outputs.npz")
    return 1 / (1 + np.exp(-z["logits"] / RESULTS[name].get("temperature", 1.0)))

def ens_dev(members):
    p = np.mean([run_probs(n, "dev") for n in members], 0)
    return multilabel_scores(Y_DEV, decide(p, tune_global(Y_DEV, p)))["macro_f1"]

In [ ]:
SUB_DIR = OUTPUT_DIR / "submissions"
SUB_DIR.mkdir(exist_ok=True)
VALID_IDS = set(int(x) for x in LABEL_IDS)
print(f"{'experiment':30s} {'dev macro-F1':>12s}  zip")
for n in ORDER:
    zp = OUTPUT_DIR / "runs" / n / "submission.zip"
    if not zp.exists():
        continue
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"], z.namelist()
        sub = json.loads(z.read("prediction.json"))
    assert isinstance(sub, list) and [s["id"] for s in sub] == test_ids
    for s in sub:
        assert set(s) == {"id", "pred"} and isinstance(s["pred"], list)
        assert all(isinstance(x, int) for x in s["pred"]) and len(s["pred"]) == len(set(s["pred"]))
        assert set(s["pred"]) <= VALID_IDS
    dest = SUB_DIR / f"phase5_{ARCH_NAME}_{n}.zip"
    shutil.copy(zp, dest)
    print(f"{n:30s} {RESULTS[n]['dev']['macro_f1']:12.4f}  {dest}")
print("\nEach zip: prediction.json at the root = a JSON list of {\"id\", \"pred\": [statute ids]} - integer ids from the")
print("train/dev label set, no duplicates, one entry per test document. Upload on Codabench (Phase 5); your best four")
print("submissions must come from different Phase-1 models.")